# M10.2 · Risk Score Validation vs Ground-Truth Signals (CRISP-DM Phase 5)

**What this notebook answers**

1. Do device-months in the **high** Isolation-Forest band experience more **maintenance events** in the same or following month than device-months in the **low** band?
2. What is the **lift** of the top-decile risk score over the cohort base rate?
3. Do clustering and anomaly scoring **agree** on which device-months are problematic?

**Caveats baked into the analysis (not hidden):**

- `staging.sinistre` is empty (0 rows). No accident ground truth.
- `staging.offense` has 4 rows globally. Statistically useless.
- `warehouse.fact_maintenance` has ~150 events linkable to a device in the modeling window. This is **directional evidence**, not a tight statistical claim.

That is why we report **lift** and **per-band rates**, not p-values. We also avoid claiming precision @ k for k smaller than 30.

## 1. Bootstrap (re-pull features, re-fit Isolation Forest)

In [ ]:
from __future__ import annotations
import sys, pathlib
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for c in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = c / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src)); break

import pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import matplotlib.pyplot as plt
from accent_fleet.db import get_engine
from sqlalchemy import text

FEATURES = [
    'overspeed_per_100km', 'avg_speed_over_limit', 'high_speed_trip_ratio',
    'speed_alert_per_100km',
    'harsh_brake_per_100km', 'harsh_accel_per_100km', 'harsh_corner_per_100km',
    'monthly_idle_ratio', 'high_rpm_minutes_per_day',
    'night_trip_ratio', 'rush_hour_trip_ratio',
    'stddev_trip_distance', 'short_trip_ratio',
]
ID_COLS = ['tenant_id', 'device_id', 'year_month']

with get_engine().connect() as conn:
    df = pd.read_sql(text('''
        SELECT * FROM marts.v_ml_features_full
        WHERE year_month >= '2025-01'
          AND total_distance_km >= 100
          AND total_ignition_on_minutes > 0
    '''), conn)
print('feature rows:', len(df), '|', 'tenants:', sorted(df.tenant_id.unique().tolist()))

In [ ]:
def fit_iso_one_tenant(sub: pd.DataFrame):
    X = sub[FEATURES].fillna(0).to_numpy()
    if len(X) < 50:
        return None
    Xs = StandardScaler().fit_transform(X)
    iso = IsolationForest(n_estimators=200, contamination='auto',
                          random_state=42, n_jobs=-1).fit(Xs)
    raw = -iso.decision_function(Xs)
    score = (raw - raw.min()) / (raw.max() - raw.min() + 1e-9)
    out = sub[ID_COLS].copy()
    out['risk_score'] = score
    out['risk_band'] = pd.cut(score, bins=[-0.01, 0.4, 0.7, 1.01],
                               labels=['low', 'medium', 'high'])
    return out

scores = pd.concat([r for r in (fit_iso_one_tenant(s) for _, s in df.groupby('tenant_id')) if r is not None],
                    ignore_index=True)
scores.risk_band.value_counts()

## 2. Pull maintenance ground truth

Chain: `fact_maintenance.vehicle_id` → `dim_vehicle.vehicule_id` → `dim_vehicle.vehicle_sk` → `dim_device.device_id`.

We bucket maintenance by `maintenance_date`'s YYYY-MM, then join twice:
- **Concurrent month** — risk in month *T* matches maintenance in *T*. (Risk should rise the month maintenance happens.)
- **Predictive month** — risk in month *T* matches maintenance in *T+1*. (Does today's high-risk score precede tomorrow's maintenance?)

In [ ]:
with get_engine().connect() as conn:
    maint = pd.read_sql(text('''
        SELECT dd.device_id, fm.tenant_id,
               TO_CHAR(fm.maintenance_date, 'YYYY-MM') AS year_month,
               COUNT(*) AS maint_events, SUM(fm.total_cost) AS maint_cost
        FROM warehouse.fact_maintenance fm
        JOIN warehouse.dim_vehicle dv
          ON fm.vehicle_id = dv.vehicule_id AND fm.tenant_id = dv.tenant_id
        JOIN warehouse.dim_device dd ON dv.vehicle_sk = dd.vehicle_sk
        WHERE fm.maintenance_date >= '2025-01-01'
        GROUP BY 1, 2, 3
    '''), conn)
print('maintenance device-month rows:', len(maint), '|', 'sum events:', int(maint.maint_events.sum()))
maint.head()

### 2a. Tenant-overlap diagnostic — read this before interpreting any join

Maintenance ground truth is only meaningful if the **same tenants** appear in both `marts.v_ml_features_full` and `warehouse.fact_maintenance`. We check that explicitly. If the overlap is empty, the maintenance-based gates below will all read as zero — that is a *data coverage* result, not a model failure.

In [ ]:
feat_tenants = set(scores.tenant_id.unique())
maint_tenants = set(maint.tenant_id.unique())
overlap = feat_tenants & maint_tenants
print('tenants with features:', sorted(feat_tenants))
print('tenants with maintenance:', sorted(maint_tenants))
print('overlap:', sorted(overlap))
MAINT_USABLE = bool(overlap)
if not MAINT_USABLE:
    print('\n[WARN] maintenance data and feature data are in disjoint tenants.')
    print('       Concurrent / predictive / lift gates will all be 0.')
    print('       Verdict will fall back to internal-consistency checks.')

## 3. Concurrent-month band rates

In [ ]:
joined = scores.merge(maint, on=ID_COLS, how='left')
joined['has_maint'] = joined.maint_events.fillna(0) > 0

per_band = (joined.groupby(['tenant_id', 'risk_band'], observed=True)
                  .agg(n=('device_id', 'size'),
                       n_with_maint=('has_maint', 'sum'),
                       maint_rate=('has_maint', 'mean'))
                  .round(4))
per_band

In [ ]:
# Aggregate across all tenants for the headline number
global_band = (joined.groupby('risk_band', observed=True)
                      .agg(n=('device_id', 'size'),
                           maint_rate=('has_maint', 'mean'))
                      .assign(lift=lambda x: x['maint_rate'] / x['maint_rate'].loc['low']
                                               if 'low' in x.index else np.nan)
                      .round(3))
global_band

## 4. Predictive: risk in month T → maintenance in month T+1

In [ ]:
scores_for_join = scores.copy()
scores_for_join['next_year_month'] = (
    pd.to_datetime(scores_for_join.year_month + '-01') +
    pd.offsets.MonthBegin(1)).dt.strftime('%Y-%m')

next_maint = maint.rename(columns={'year_month': 'next_year_month'})
pred = scores_for_join.merge(
    next_maint, on=['tenant_id', 'device_id', 'next_year_month'], how='left')
pred['has_next_maint'] = pred.maint_events.fillna(0) > 0

pred_band = (pred.groupby('risk_band', observed=True)
                  .agg(n=('device_id', 'size'),
                       next_month_maint_rate=('has_next_maint', 'mean'))
                  .round(4))
pred_band

## 5. Top-decile lift

A common executive question: *"how much more likely is a maintenance event in the riskiest 10% of device-months vs the average?"* This is the operational metric we'd put on a dashboard.

In [ ]:
joined['decile'] = (joined.groupby('tenant_id', group_keys=False)
                           .risk_score.transform(lambda s: pd.qcut(s, 10, labels=False, duplicates='drop')))
base_rate = joined.has_maint.mean()
top_rate = joined.loc[joined.decile == 9, 'has_maint'].mean()
lift = top_rate / base_rate if base_rate > 0 else float('nan')
print(f'base maintenance rate: {base_rate:.3%}')
print(f'top-decile maintenance rate: {top_rate:.3%}')
print(f'lift @ top decile: {lift:.2f}x  (n top decile = {(joined.decile==9).sum()})')

## 6. Cluster ↔ Risk-band agreement

If clustering and anomaly scoring identify *roughly the same* device-months as problematic, the two methods reinforce each other. If they disagree wildly, one of them is wrong (or they are picking up different definitions of risk).

In [ ]:
def fit_km_one_tenant(sub):
    X = sub[FEATURES].fillna(0).to_numpy()
    if len(X) < 50: return None
    Xs = StandardScaler().fit_transform(X)
    best = None
    for k in range(3, 7):
        km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xs)
        sil = silhouette_score(Xs, km.labels_)
        if best is None or sil > best['sil']:
            best = {'k': k, 'sil': sil, 'labels': km.labels_}
    out = sub[ID_COLS].copy(); out['cluster'] = best['labels']
    return out

clusters = pd.concat([r for r in (fit_km_one_tenant(s) for _, s in df.groupby('tenant_id')) if r is not None],
                     ignore_index=True)
agree = clusters.merge(scores, on=ID_COLS, how='inner')

# For each tenant, find the 'risk-leaning' cluster (highest mean risk_score)
# and check what fraction of its members fall in the high band.
agg_rows = []
for t, sub in agree.groupby('tenant_id'):
    cl_mean = sub.groupby('cluster').risk_score.mean()
    riskiest = cl_mean.idxmax()
    mask = sub.cluster == riskiest
    agg_rows.append({
        'tenant_id': t, 'riskiest_cluster': int(riskiest),
        'n_in_cluster': int(mask.sum()),
        'mean_risk_in_cluster': round(float(cl_mean.loc[riskiest]), 3),
        'frac_high_band': round(float((sub.loc[mask, 'risk_band'] == 'high').mean()), 3),
        'frac_high_band_overall': round(float((sub.risk_band == 'high').mean()), 3),
    })
pd.DataFrame(agg_rows)

## 7. Internal-consistency fallback (used when external ground truth is unavailable)

If the maintenance overlap is empty, the only legitimate evidence we have is **internal consistency**: do the high-band rows actually look riskier on the *input features themselves*? We compute, per tenant, the ratio of mean overspeed/harsh/idle in the **high** band vs the **low** band. A working anomaly score should produce ratios > 1 on the unsafe-direction features, even without a label.

In [ ]:
unsafe_features = ['overspeed_per_100km', 'speed_alert_per_100km',
                   'harsh_brake_per_100km', 'harsh_accel_per_100km',
                   'harsh_corner_per_100km', 'high_rpm_minutes_per_day',
                   'monthly_idle_ratio']
scored_with_features = scores.merge(df[ID_COLS + unsafe_features], on=ID_COLS, how='left')
ic_rows = []
for t, sub in scored_with_features.groupby('tenant_id'):
    high = sub[sub.risk_band == 'high'][unsafe_features].mean()
    low = sub[sub.risk_band == 'low'][unsafe_features].mean()
    ratios = (high / low.replace(0, np.nan)).round(2)
    ic_rows.append({'tenant_id': t,
                    'features_with_ratio_gt_1': int((ratios > 1).sum()),
                    'features_with_ratio_gt_2': int((ratios > 2).sum()),
                    'mean_ratio': round(float(ratios.mean(skipna=True)), 2)})
internal_consistency = pd.DataFrame(ic_rows)
internal_consistency

## 8. Verdict

We treat any of the following as *passing evidence* for the risk score:

- **External (preferred):** the **high** band has maintenance rate >= **2x** the **low** band (concurrent or predictive); top decile lift >= **1.5x**.
- **Internal (fallback when ground truth missing):** in every tenant, **>= 5 of 7** unsafe features show high-band mean > low-band mean, and **>= 2** show >= 2x ratio.
- Cluster ↔ risk-band agreement: the riskiest cluster shows `frac_high_band` >= 2x `frac_high_band_overall` in at least one tenant.

In [ ]:
# External gates (will be NaN/0 when MAINT_USABLE is False)
try:
    ratio_concurrent = (global_band.loc['high', 'maint_rate']
                        / max(global_band.loc['low', 'maint_rate'], 1e-9))
except KeyError:
    ratio_concurrent = float('nan')
try:
    ratio_predictive = (pred_band.loc['high', 'next_month_maint_rate']
                        / max(pred_band.loc['low', 'next_month_maint_rate'], 1e-9))
except KeyError:
    ratio_predictive = float('nan')

# Internal-consistency gate
ic_pass = ((internal_consistency.features_with_ratio_gt_1 >= 5).all()
           and (internal_consistency.features_with_ratio_gt_2 >= 2).all())

# Cluster <-> band agreement
agg_df = pd.DataFrame(agg_rows)
cluster_agreement = ((agg_df.frac_high_band >= 2 * agg_df.frac_high_band_overall).any())

verdict = pd.DataFrame.from_dict({
    'external | concurrent high/low maint ratio (>=2)': [round(ratio_concurrent, 2), ratio_concurrent >= 2],
    'external | predictive high/low maint ratio (>=2)': [round(ratio_predictive, 2), ratio_predictive >= 2],
    'external | top-decile lift (>=1.5)': [round(lift, 2) if not np.isnan(lift) else float('nan'), (lift >= 1.5) if not np.isnan(lift) else False],
    'internal | unsafe features gate (>=5 ratios>1, >=2 ratios>2 per tenant)': [bool(ic_pass), bool(ic_pass)],
    'cluster<->band agreement (>=2x)': [bool(cluster_agreement), bool(cluster_agreement)],
}, orient='index', columns=['value', 'pass'])
verdict

**Go / No-go.**

- If any of the **external** rows pass → ship as a directional risk indicator without caveats.
- If only the **internal-consistency** + **cluster-agreement** rows pass (and external are NaN due to no overlap) → ship as a **provisional** indicator with an explicit dashboard footer:  *"validated against feature-space anomalies, not against incident outcomes — outcome backtesting blocked by source data coverage in `staging.maintenance` for the modeled tenants."*
- If neither external nor internal gates pass → do not put `risk_score` in front of a customer; surface only cluster personas.

**Next action.** Run `03_stability_and_fairness.ipynb` to confirm the score is stable month-over-month and balanced across tenants before any production deployment.